In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
df = pd.read_excel("../data/raw/online_retail_II.xlsx")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB


In [10]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

In [25]:
df[df['Customer ID'].isna()].sample(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
30865,491969,22246,"GARLAND, MAGIC GARDEN 1.8M",1,2009-12-14 17:57:00,4.30,NaN,United Kingdom
5627,489857,20682,RED SPOTTY CHILDS UMBRELLA,3,2009-12-02 14:43:00,6.91,NaN,United Kingdom
143247,502976,22090,PAPER BUNTING RETRO SPOTS,1,2010-03-29 14:54:00,5.91,NaN,United Kingdom
518195,537638,22108,PING! MICROWAVE PLATE,1,2010-12-07 15:28:00,3.36,NaN,United Kingdom
173053,505768,79323W,NaN,-136,2010-04-26 10:27:00,0.00,NaN,United Kingdom
147636,503428,85089,CANDY SPOT BUNNY,1,2010-03-31 17:31:00,5.91,NaN,United Kingdom
518467,537638,72807A,SET/3 ROSE CANDLE IN JEWELLED BOX,3,2010-12-07 15:28:00,8.47,NaN,United Kingdom
42884,493073,20961,STRAWBERRY BATH SPONGE,1,2009-12-22 09:41:00,2.57,NaN,United Kingdom
322758,520833,21879,HEARTS GIFT TAPE,1,2010-08-31 12:41:00,1.66,NaN,United Kingdom
147665,503428,20971,PINK BLUE FELT CRAFT TRINKET BOX,1,2010-03-31 17:31:00,2.51,NaN,United Kingdom


In [26]:
df[df['Quantity'] < 0]['Invoice'].str.startswith('C').value_counts()

Invoice
True    10205
Name: count, dtype: int64

In [33]:
df[df['Quantity'] < 0]['Quantity'].value_counts()

Quantity
-1       4713
-2       1578
-3        792
-4        643
-12       643
         ... 
-324        1
-4354       1
-575        1
-490        1
-364        1
Name: count, Length: 375, dtype: int64

In [35]:
df[df['Price'] == 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
525231,538159,21324,NaN,-18,2010-12-09 17:17:00,0.0,NaN,United Kingdom
525232,538158,20892,NaN,-32,2010-12-09 17:17:00,0.0,NaN,United Kingdom
525233,538160,20956,NaN,288,2010-12-09 17:18:00,0.0,NaN,United Kingdom
525234,538161,46000S,Dotcom sales,-100,2010-12-09 17:25:00,0.0,NaN,United Kingdom


In [38]:
df[df['Price'] == 0]['Customer ID'].isna().sum()

np.int64(3656)

## Решения по очистке данных

**Пропуски в Customer ID (107 927 строк):**
Удаляем — без ID клиента невозможно посчитать RFM метрики.

**Отрицательный Quantity (возвраты):**
Оставляем — учитываем при подсчёте Monetary чтобы не завышать реальную ценность клиента.

**Пропуски в Description (2928 строк):**
Игнорируем — Description не используется в RFM метриках.

**Price = 0 (3687 строк):**
Удаляем — большинство уже не имеют Customer ID, 
оставшиеся дадут 0 в Monetary и исказят RFM.